<a href="https://colab.research.google.com/github/Abdulkadir03/Crop-Recommender-for-Sahel-Region/blob/main/notebooks/02_climate_features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Climate features

**Purpose:** Add long-term climate features to every grid cell from notebook 01, for use as model inputs.

**Inputs:**
- `grid.parquet` from notebook 01 (~2 km cells, nine Sahel countries)
- Google Earth Engine: `UCSB-CHG/CHIRPS/PENTAD` (rainfall), `IDAHO_EPSCOR/TERRACLIMATE` (temperature, PET, water balance), `ECMWF/ERA5_LAND/MONTHLY_AGGR` (temperature, soil water, precipitation cross-check)
- Requires an Earth Engine account and a Cloud project (set `PROJECT_ID` to your own)

**Method:** Features are computed on Earth Engine and sampled at each grid point, exported as small CSV tiles, then merged here.
- **Baseline window:** 1991-2020 (30 years). **Recent window:** 2018-2024, used for the rainfall anomaly.
- **Rainfall (CHIRPS):** mean annual, year-to-year variability (CV), dry-year (10th percentile), June-September mean, 30-year trend, recent mean and anomaly (%)
- **Temperature and water balance (TerraClimate):** mean, hottest-month and coldest-month temperature, PET, climatic water deficit, actual ET, soil moisture, VPD, solar radiation, growing months (rain > 50% of PET), months above 50 mm
- **ERA5-Land:** mean temperature, June-September soil water (two layers), annual precipitation

**Output:** `climate.parquet`, one row per `grid_id` with 22 float32 features. Stored in Google Drive, not in this repo.

**Notes:**
- Source resolutions are coarse (CHIRPS ~5 km, TerraClimate ~4 km, ERA5-Land ~11 km), so neighbouring 2 km cells share values. Spatial cross-validation blocks must be much larger than ~10 km, or train and test cells will share identical climate values and scores will be inflated.
- 77 Grid cells with no climate value (e.g. coastal cells) are reported at the end of the merge step.
- Daily-based features (dry spells, rainy-season onset) are not included here.

In [ ]:
# @title Cell 1: setup and load the grid
import ee, os, glob
import pandas as pd
from google.colab import drive

PROJECT_ID = 'your-project-id'        # your Cloud project ID

BASELINE = ('1991-01-01', '2021-01-01')   # end date exclusive
RECENT   = ('2018-01-01', '2025-01-01')
GRID_SCALE = 2000
OUT_FOLDER = 'crop_project_climate_feature'       # new Drive folder for this notebook's exports

ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

drive.mount('/content/drive')
BASE = '/content/drive/MyDrive'
grid = pd.read_parquet(f'{BASE}/crop_project/grid.parquet')
print(grid.shape)
print(grid.groupby('adm0').size())

In [ ]:
# @title Cell 2: check the datasets and band names
def info(cid):
    col = ee.ImageCollection(cid)
    start = ee.Date(col.aggregate_min('system:time_start')).format('YYYY-MM-dd').getInfo()
    end = ee.Date(col.aggregate_max('system:time_start')).format('YYYY-MM-dd').getInfo()
    print(cid, '|', start, '->', end)
    print('   bands:', col.first().bandNames().getInfo())

for cid in ['UCSB-CHG/CHIRPS/PENTAD',
            'IDAHO_EPSCOR/TERRACLIMATE',
            'ECMWF/ERA5_LAND/MONTHLY_AGGR']:
    info(cid)

In [ ]:
# @title Cell 3: build the climate feature image
y0, y1 = int(BASELINE[0][:4]), int(BASELINE[1][:4]) - 1      # 1991 .. 2020
n_years = y1 - y0 + 1
years = ee.List.sequence(y0, y1)

# ---------- CHIRPS: rainfall ----------
chirps = ee.ImageCollection('UCSB-CHG/CHIRPS/PENTAD').select('precipitation')

def yearly(y):
    y = ee.Number(y)
    start = ee.Date.fromYMD(y, 1, 1)
    annual = chirps.filterDate(start, start.advance(1, 'year')).sum().rename('annual')
    jjas = chirps.filterDate(ee.Date.fromYMD(y, 6, 1),
                             ee.Date.fromYMD(y, 10, 1)).sum().rename('jjas')
    yr = ee.Image.constant(y).float().rename('year')
    return annual.addBands(jjas).addBands(yr).set('year', y)

yc = ee.ImageCollection(years.map(yearly))
ann = yc.select('annual')

ann_mean = ann.mean().rename('chirps_annual_mean')
ann_cv   = ann.reduce(ee.Reducer.stdDev()).divide(ann_mean).rename('chirps_annual_cv')
ann_p10  = ann.reduce(ee.Reducer.percentile([10])).rename('chirps_annual_p10')
jjas_mean = yc.select('jjas').mean().rename('chirps_jjas_mean')
trend = (yc.select(['year', 'annual']).reduce(ee.Reducer.linearFit())
         .select('scale').rename('chirps_trend_mm_per_yr'))

n_recent = int(RECENT[1][:4]) - int(RECENT[0][:4])
recent_mean = (chirps.filterDate(*RECENT).sum().divide(n_recent)
               .rename('chirps_annual_recent'))
recent_anom = recent_mean.divide(ann_mean).subtract(1).multiply(100).rename('chirps_recent_anom_pct')

# ---------- TerraClimate ----------
tc = ee.ImageCollection('IDAHO_EPSCOR/TERRACLIMATE').filterDate(*BASELINE)

temp_mean = (tc.map(lambda im: im.select('tmmn').add(im.select('tmmx'))
                    .divide(2).multiply(0.1).rename('t')).mean().rename('temp_mean'))
pet_annual = tc.select('pet').sum().divide(n_years).multiply(0.1).rename('pet_annual')
def_annual = tc.select('def').sum().divide(n_years).multiply(0.1).rename('def_annual')
aet_annual = tc.select('aet').sum().divide(n_years).multiply(0.1).rename('aet_annual')
soil_mean  = tc.select('soil').mean().multiply(0.1).rename('soil_moisture_mean')
vpd_mean   = tc.select('vpd').mean().multiply(0.01).rename('vpd_mean')
srad_mean  = tc.select('srad').mean().multiply(0.1).rename('srad_mean')

# Monthly climatology (12 images) for seasonality features
def month_clim(m):
    m = ee.Number(m)
    return tc.filter(ee.Filter.calendarRange(m, m, 'month')).mean().set('month', m)

clim = ee.ImageCollection(ee.List.sequence(1, 12).map(month_clim))
tmax_peak = clim.select('tmmx').max().multiply(0.1).rename('tmax_hottest_month')
tmin_cold = clim.select('tmmn').min().multiply(0.1).rename('tmin_coldest_month')
# growing months: rainfall > 50% of PET (PET stored x10, so 0.5 * 0.1 = 0.05)
growing = (clim.map(lambda im: im.select('pr').gt(im.select('pet').multiply(0.05)).rename('w'))
           .sum().rename('growing_months'))
wet50 = clim.map(lambda im: im.select('pr').gt(50).rename('w')).sum().rename('wet_months_50mm')

# ---------- ERA5-Land ----------
era = ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR').filterDate(*BASELINE)
era_t = era.select('temperature_2m').mean().subtract(273.15).rename('era5_temp_mean')
era_sw = (era.filter(ee.Filter.calendarRange(6, 9, 'month'))
          .select(['volumetric_soil_water_layer_1', 'volumetric_soil_water_layer_2']).mean()
          .rename(['era5_soilwater_l1_jjas', 'era5_soilwater_l2_jjas']))
era_p = (era.select('total_precipitation_sum').sum().divide(n_years)
         .multiply(1000).rename('era5_precip_annual'))

climate_img = ee.Image.cat([
    ann_mean, ann_cv, ann_p10, jjas_mean, trend, recent_mean, recent_anom,
    temp_mean, tmax_peak, tmin_cold, pet_annual, def_annual, aet_annual,
    soil_mean, vpd_mean, srad_mean, growing, wet50,
    era_t, era_sw, era_p]).float()

print(climate_img.bandNames().getInfo())

In [ ]:
# @title Cell 4: test on a few real points
def to_fc(d):
    return ee.FeatureCollection([
        ee.Feature(ee.Geometry.Point([float(r.lon), float(r.lat)]), {'grid_id': r.grid_id})
        for r in d.itertuples()])

test_pts = grid.sample(6, random_state=0)
res = climate_img.sampleRegions(collection=to_fc(test_pts), properties=['grid_id'],
                                scale=GRID_SCALE, tileScale=4).getInfo()
out = pd.DataFrame([f['properties'] for f in res['features']]).set_index('grid_id')
out.insert(0, 'adm0', test_pts.set_index('grid_id').adm0)
pd.set_option('display.width', 200)
print(out.T.round(2))

In [ ]:
# @title Cell 5: start the exports
BLOCK = 2          # degrees per block
MAXPTS = 8000      # max points per task

g = grid[['grid_id', 'lon', 'lat']].copy()
g['bx'] = (g.lon // BLOCK * BLOCK).astype(int)
g['by'] = (g.lat // BLOCK * BLOCK).astype(int)

jobs = []
for (bx, by), d in g.groupby(['bx', 'by']):
    for i in range(0, len(d), MAXPTS):
        jobs.append((f'clim_x{bx}_y{by}_p{i // MAXPTS}', d.iloc[i:i + MAXPTS]))
print(len(jobs), 'export jobs for', len(g), 'points')

already = {t.config.get('description') for t in ee.batch.Task.list()[:2000]}
started = 0
for name, d in jobs:
    if name in already:
        continue
    ee.batch.Export.table.toDrive(
        collection=climate_img.sampleRegions(collection=to_fc(d), properties=['grid_id'],
                                             scale=GRID_SCALE, tileScale=4),
        description=name, folder=OUT_FOLDER, fileFormat='CSV').start()
    started += 1
print('Started', started, 'tasks')

In [ ]:
# @title Cell 6: monitor
tasks = [t for t in ee.batch.Task.list()[:2000]
         if t.config.get('description', '').startswith('clim_')]
counts = {}
for t in tasks:
    s = t.status()
    counts[s['state']] = counts.get(s['state'], 0) + 1
    if s['state'] == 'FAILED':
        print('FAILED:', s['description'], '->', s.get('error_message'))
print(counts)

In [ ]:
# @title Cell 7: merge into climate.parquet. Run this once every task shows COMPLETED.
files = glob.glob(f'{BASE}/{OUT_FOLDER}/*.csv')
parts = [pd.read_csv(f) for f in files if os.path.getsize(f) > 0]
clim = pd.concat(parts, ignore_index=True)
clim = clim.drop(columns=['system:index', '.geo'], errors='ignore')
clim = clim.drop_duplicates('grid_id').reset_index(drop=True)
print(len(files), 'files ->', clim.shape)

# Which grid cells got no climate values (masked pixels, e.g. coastal cells)?
missing = grid[~grid.grid_id.isin(clim.grid_id)]
print(f'{len(missing)} of {len(grid)} grid cells have no climate row')
print(missing.groupby('adm0').size())

feat_cols = [c for c in clim.columns if c != 'grid_id']
clim[feat_cols] = clim[feat_cols].astype('float32')
clim.to_parquet(f'{BASE}/crop_project/climate.parquet', index=False)

print(clim[feat_cols].describe(percentiles=[.05, .5, .95]).T.round(2))
print('\nNaNs per column:'); print(clim[feat_cols].isna().sum())

